# OpenAPI와 TS SDK

Pydantic과 FastAPI 선언이 OpenAPI 스키마가 되고 TypeScript 클라이언트 생성 입력이 되는 과정을 확인합니다.

## 실행 환경 준비

현재 폴더나 상위 폴더에서 `labs/part7/ch30-fastapi-serving`을 찾아 작업 폴더로 사용합니다. 비밀 키를 출력하지 않습니다.

### 이 준비 셀이 하는 일

이 준비 셀은 `Path.cwd().resolve()`로 현재 폴더를 절대 경로로 읽고, 현재 폴더와 부모 폴더에서 `agent_service`가 있는 첫 위치를 `next(..., None)`로 찾습니다. 찾지 못하면 `assert`가 실행을 중단합니다. 찾은 뒤 `os.chdir(lab)`이 이후 import와 파일 접근의 기준 폴더를 바꾸고, 마지막 출력은 실제 선택된 폴더 이름을 확인합니다.

In [1]:
import os
from pathlib import Path
start = Path.cwd().resolve()
lab = next((p for p in (start, *start.parents)
    if (p / 'agent_service').is_dir()), None)
assert lab is not None, 'agent_service가 있는 Ch30 실습 폴더에서 실행하세요.'
os.chdir(lab)
print('실습 폴더:', Path.cwd().name)

실습 폴더: ch30-fastapi-serving


## 스키마 생성

서버 프로세스를 시작하지 않아도 앱 객체에서 같은 OpenAPI 문서를 얻을 수 있습니다.

### 바로 다음 코드를 읽는 방법

`app.openapi()`는 현재 라우트와 Pydantic 모델에서 OpenAPI 문서를 파이썬 사전으로 만듭니다. `paths`에는 URL별 연산이, `$ref`에는 공통 스키마를 가리키는 참조가 들어갑니다. 예제는 요청이 `ChatRequest`, 200 응답이 `ChatResponse`를 참조하는지 검사하고 JSON 파일로 저장합니다. SDK 생성기는 실행 중인 서버를 추측하는 대신 이 문서를 입력으로 사용합니다.

In [2]:
import json
from agent_service.app import app
schema = app.openapi()
path = Path('openapi.json')
path.write_text(json.dumps(schema, ensure_ascii=False, indent=2), encoding='utf-8')
print(schema['openapi'], len(schema['paths']))
assert '/v1/chat' in schema['paths']

3.1.0 5


## 타입과 보안 정보 관찰

요청 스키마의 필수 필드와 응답 타입이 계약 문서에 포함됩니다.

In [3]:
chat = schema['paths']['/v1/chat']['post']
request_ref = chat['requestBody']['content']['application/json']['schema']['$ref']
response_ref = chat['responses']['200']['content']['application/json']['schema']['$ref']
print(request_ref, response_ref)
assert request_ref.endswith('/ChatRequest')
assert response_ref.endswith('/ChatResponse')

#/components/schemas/ChatRequest #/components/schemas/ChatResponse


## 반복 실험과 결과 해석

같은 앱에서 두 번 생성한 스키마를 정렬해 직렬화하면 같은 결과가 나옵니다.

### 이 단계에서 구분할 범위

스키마가 같은 문자열로 두 번 생성된다는 검사는 현재 프로세스에서 결정적인 출력을 확인할 뿐, 이전 배포와의 호환성을 증명하지 않습니다. 필수 필드 추가, 타입 축소, 상태 코드 삭제는 기존 클라이언트를 깨뜨릴 수 있습니다. 반대로 선택 필드 추가도 생성된 SDK를 다시 배포해야 소비자가 타입으로 사용할 수 있습니다.

In [4]:
first = json.dumps(app.openapi(), sort_keys=True)
second = json.dumps(app.openapi(), sort_keys=True)
print('동일:', first == second, '문자 수:', len(first))
assert first == second
print('SDK 생성 입력:', path.resolve())

동일: True 문자 수: 4885
SDK 생성 입력: [사용자 폴더]/working/claude/projects/한국경제신문/labs/part7/ch30-fastapi-serving/openapi.json
